# Week 5 Exercise – RAG Knowledge Worker (iyanuashiri)

Chat with your knowledge base: **upload** files in the UI → **ingest** (chunk, embed, Chroma) → **retrieve** top-k → **answer** via OpenRouter with **chat history** and **sources**. Uploaded files are used only for indexing (Gradio temp files); no folder path in the codebase.

## Imports and environment

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(override=True)
try:
    from decouple import config
    OPEN_ROUTER_API_KEY = config("OPEN_ROUTER_API_KEY")
except Exception:
    OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")

OPEN_ROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=OPEN_ROUTER_BASE_URL, api_key=OPEN_ROUTER_API_KEY)

DB_NAME = "week5_rag_db"
DB_NAME_NEW = "week5_rag_db_new"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MAX_CHAT_TURNS = 10

MODELS = [
    "google/gemini-3.1-pro-preview",
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = None

def _get_db_path():
    if os.path.exists(DB_NAME):
        return DB_NAME
    if os.path.exists(DB_NAME_NEW):
        return DB_NAME_NEW
    return None

_db_path = _get_db_path()
if _db_path:
    vectorstore = Chroma(persist_directory=_db_path, embedding_function=embeddings)

## Ingest: load → chunk → embed → Chroma

In [ ]:
def load_docs_from_files(file_paths) -> list:
    """Load from list of file paths (Gradio uploads; temp files). Supports .pdf, .txt, .md."""
    if not file_paths:
        return []
    if not isinstance(file_paths, list):
        file_paths = [file_paths]
    docs = []
    for p in file_paths:
        path = Path(p) if isinstance(p, str) else Path(getattr(p, "name", str(p)))
        if not path.exists():
            continue
        suffix = path.suffix.lower()
        try:
            if suffix == ".pdf":
                loader = PyPDFLoader(str(path))
                docs.extend(loader.load())
            elif suffix in (".txt", ".md"):
                loader = TextLoader(str(path), encoding="utf-8")
                docs.extend(loader.load())
        except Exception:
            pass
    return docs

def ingest(uploaded_files, replace: bool = True) -> str:
    """Ingest from uploaded files only: load → chunk → embed → Chroma. Returns status message."""
    global vectorstore
    docs = load_docs_from_files(uploaded_files)
    if not docs:
        return "No documents loaded. Upload one or more files (.pdf, .txt, .md), then click Ingest."
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(docs)
    vectorstore = None
    import shutil
    persist_dir = DB_NAME
    if replace:
        try:
            if os.path.exists(DB_NAME):
                shutil.rmtree(DB_NAME)
            if os.path.exists(DB_NAME_NEW):
                shutil.rmtree(DB_NAME_NEW)
            persist_dir = DB_NAME
        except PermissionError:
            persist_dir = DB_NAME_NEW
            if os.path.exists(DB_NAME_NEW):
                try:
                    shutil.rmtree(DB_NAME_NEW)
                except PermissionError:
                    pass
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_dir,
    )
    return f"Ingested {len(docs)} documents, {len(chunks)} chunks."

## RAG: retrieve → build context → OpenRouter chat (with history and sources)

In [ ]:
SYSTEM_PROMPT = """You are a helpful assistant that answers questions using ONLY the provided context. If the context does not contain enough information, say so. Do not make up facts."""

def build_sources_text(docs) -> str:
    seen = set()
    lines = []
    for i, doc in enumerate(docs, 1):
        src = doc.metadata.get("source", "unknown")
        if src not in seen:
            seen.add(src)
            preview = (doc.page_content or "")[:200].replace("\n", " ")
            lines.append(f"{i}. {src}\n   {preview}...")
    return "\n\n".join(lines) if lines else "No sources."

def answer_with_rag(
    message: str,
    history: list,
    k: int,
    model: str,
) -> tuple[list, str, str]:
    """Handle one user message: retrieve k docs, call OpenRouter with history, return (updated_history, answer, sources_text)."""
    if vectorstore is None:
        return history + [[message, "No index yet. Upload files and click **Ingest** first."]], "", ""
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(message)
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)
    sources_text = build_sources_text(docs)

    messages = [{"role": "system", "content": SYSTEM_PROMPT + "\n\nContext:\n" + context}]
    for h in history[-MAX_CHAT_TURNS:]:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})

    try:
        resp = openrouter.chat.completions.create(model=model, messages=messages)
        answer = (resp.choices[0].message.content or "").strip()
    except Exception as e:
        answer = f"Error calling OpenRouter: {e}"
    new_history = history + [[message, answer]]
    return new_history, answer, sources_text

## Gradio UI: Ingest + Chat + Sources

In [ ]:
def run_ingest(files) -> str:
    return ingest(files, replace=True)

def chat_turn(message: str, history: list, k: int, model: str) -> tuple[list, list, str, str]:
    if not (message or message.strip()):
        return history, history, "", ""
    new_history, _answer, sources_text = answer_with_rag(message.strip(), history, k, model)
    return new_history, new_history, sources_text, ""

In [ ]:
# Ingest is upload-only; no folder path. Gradio temp files are used and can be cleared when the app closes.

In [ ]:
with gr.Blocks(title="Week 5 RAG – Knowledge Worker") as app:
    gr.Markdown("## RAG Knowledge Worker – Ingest & Chat")
    chat_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Ingest (upload only)")
            file_in = gr.File(file_count="multiple", label="Upload files (.pdf, .txt, .md)")
            ingest_btn = gr.Button("Ingest (build/replace index)")
            ingest_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### Chat")
            chatbot = gr.Chatbot(label="Chat", height=400)
            msg_in = gr.Textbox(label="Question", placeholder="Ask about your documents...", lines=2)
            with gr.Row():
                k_slider = gr.Slider(1, 15, value=5, step=1, label="Retrieve top k chunks")
                model_dd = gr.Dropdown(choices=MODELS, value=MODELS[0], label="Model")
            submit_btn = gr.Button("Submit")
            gr.Markdown("### Sources (chunks used for last answer)")
            sources_out = gr.Textbox(label="Sources", lines=6, interactive=False)

    ingest_btn.click(fn=run_ingest, inputs=[file_in], outputs=ingest_status)
    submit_btn.click(
        fn=chat_turn,
        inputs=[msg_in, chat_state, k_slider, model_dd],
        outputs=[chat_state, chatbot, sources_out, msg_in],
    )
    msg_in.submit(
        fn=chat_turn,
        inputs=[msg_in, chat_state, k_slider, model_dd],
        outputs=[chat_state, chatbot, sources_out, msg_in],
    )

app.launch()